# Gold · Resumen por Pedido

## Propósito

Construir la tabla base de todos los indicadores, con un registro por pedido vigente y las métricas ya calculadas, de modo que las cuatro vistas del reporte partan de las mismas cifras.

## Flujo

1. Agregar el detalle a nivel de pedido: ventas, utilidad, beneficio promedio, cantidad y número de líneas.
2. Unir los pedidos vigentes con la versión actual del cliente, la ubicación y el modo de envío.
3. Calcular las columnas del reporte.

## Destino

Capa Gold.

**Tabla:** `gld_dataco.supply_chain.desc_ft_order_summary`

**Grano:** un registro por pedido vigente

**Escritura:** reemplazo completo en cada ejecución (`overwrite`)

## Columnas calculadas

* `es_entrega_a_tiempo`
* `shipping_variance`
* `profit_margin_pct`

## Resultado esperado

La base única de las vistas Resumen Ejecutivo, Detalle Región, Detalle Orden y Alertas.

In [0]:
# ============================================================
# GLD_ORDER_SUMMARY — Celda 1: Parámetros
# ============================================================

dbutils.widgets.text("gold_catalog", "gld_dataco", "Catálogo Gold")
dbutils.widgets.text("gold_schema", "supply_chain", "Esquema Gold")

gold_catalog = dbutils.widgets.get("gold_catalog")
gold_schema = dbutils.widgets.get("gold_schema")

GOLD_FACT_ORDERS_FQN = f"{gold_catalog}.{gold_schema}.desc_ft_orders"
GOLD_FACT_ORDER_DETAILS_FQN = f"{gold_catalog}.{gold_schema}.desc_ft_order_details"
GOLD_DIM_CUSTOMERS_FQN = f"{gold_catalog}.{gold_schema}.desc_dim_customers"
GOLD_DIM_LOCATIONS_FQN = f"{gold_catalog}.{gold_schema}.desc_dim_locations"
GOLD_DIM_SHIPPING_MODE_FQN = f"{gold_catalog}.{gold_schema}.desc_dim_shipping_mode"
GOLD_ORDER_SUMMARY_FQN = f"{gold_catalog}.{gold_schema}.desc_ft_order_summary"

print(f"✓ Destino: {GOLD_ORDER_SUMMARY_FQN}")

In [0]:
# ============================================================
# GLD_ORDER_SUMMARY — Celda 2: INICIO DE LOG DE EJECUCIÓN
# ============================================================

import uuid

id_ejecucion = dbutils.jobs.taskValues.get(
    taskKey="prepare_pipeline_control",
    key="id_ejecucion",
    debugValue=""
)

LOG_ID = str(uuid.uuid4())
NOTEBOOK = "gld_order_summary"
CAPA = "gold"

spark.sql(f"""
INSERT INTO metadata_dataco.supply_chain.hd_etl_log
(log_id, id_ejecucion, notebook, capa, estado, fecha_inicio)
VALUES ('{LOG_ID}', '{id_ejecucion}', '{NOTEBOOK}', '{CAPA}', 'EN_PROCESO', current_timestamp())
""")

dbutils.jobs.taskValues.set(key="log_id", value=LOG_ID)

print(f"✓ Registro EN_PROCESO creado en etl_log: {LOG_ID}")

In [0]:
# ============================================================
# GLD_ORDER_SUMMARY — Celda 3: Agregar detalle a nivel de pedido
# ============================================================

from pyspark.sql import functions as F

df_detalle_agregado = (
    spark.table(GOLD_FACT_ORDER_DETAILS_FQN)
    .groupBy("orden_key")
    .agg(
        F.sum("sales").alias("total_sales"),
        F.sum("order_item_total").alias("total_order_item"),
        F.sum("order_profit_per_order").alias("total_profit"),
        F.avg("benefit_per_order").alias("avg_benefit_per_order"),
        F.sum("order_item_quantity").alias("total_quantity"),
        F.countDistinct("order_item_id").alias("total_lineas")
    )
)

print(f"✓ Pedidos con detalle agregado: {df_detalle_agregado.count()}")

In [0]:
# ============================================================
# GLD_ORDER_SUMMARY — Celda 4: Unir contra fact_orders + dimensiones de contexto
# ============================================================

df_orders_vigente = (
    spark.table(GOLD_FACT_ORDERS_FQN)
    .filter(F.col("es_vigente") == True)
)

df_dim_customers = (
    spark.table(GOLD_DIM_CUSTOMERS_FQN)
    .filter(F.col("es_vigente") == True)
    .select("cliente_key", "customer_segment")
)

df_dim_locations = spark.table(GOLD_DIM_LOCATIONS_FQN).select(
    "location_key", "market", "order_region", "grupo_mercado"
)

df_dim_shipping_mode = spark.table(GOLD_DIM_SHIPPING_MODE_FQN).select(
    "shipping_mode_key", "shipping_mode"
)

df_order_summary = (
    df_orders_vigente.alias("o")
    .join(df_detalle_agregado.alias("d"), "orden_key", "left")
    .join(df_dim_customers.alias("c"), "cliente_key", "left")
    .join(df_dim_locations.alias("l"), "location_key", "left")
    .join(df_dim_shipping_mode.alias("sm"), "shipping_mode_key", "left")
    .select(
        F.col("o.orden_key"),
        F.col("o.order_id"),
        F.col("o.order_status"),
        F.col("o.delivery_status"),
        F.col("o.late_delivery_risk_original"),
        F.col("o.late_delivery_risk_calculado"),
        F.col("o.riesgo_validado"),
        F.col("o.days_for_shipping_real"),
        F.col("o.days_for_shipment_scheduled"),
        F.col("o.es_sospechoso_fraude"),
        F.col("o.es_excluido_otd_leadtime"),
         F.col("c.cliente_key"),
        F.col("c.customer_segment"),
        F.col("l.market"),
        F.col("l.order_region"),
        F.col("l.grupo_mercado"),
        F.col("sm.shipping_mode"),
        F.coalesce(F.col("d.total_sales"), F.lit(0.0)).alias("total_sales"),
        F.coalesce(F.col("d.total_profit"), F.lit(0.0)).alias("total_profit"),
        F.coalesce(F.col("d.avg_benefit_per_order"), F.lit(0.0)).alias("avg_benefit_per_order"),
        F.coalesce(F.col("d.total_quantity"), F.lit(0)).alias("total_quantity"),
        F.col("o.fecha_inicio_corte"),
        F.col("o.fecha_fin_corte"),
        F.col("o.fecha_carga"),
        F.col("o.id_ejecucion")
    )
    .withColumn(
        "es_entrega_a_tiempo",
        (F.col("days_for_shipping_real") <= F.col("days_for_shipment_scheduled")).cast("int")
    )
    .withColumn(
        "shipping_variance",
        F.col("days_for_shipping_real") - F.col("days_for_shipment_scheduled")
    )
    .withColumn(
        "profit_margin_pct",
        F.when(F.col("total_sales") > 0, F.col("total_profit") / F.col("total_sales") * 100)
         .otherwise(F.lit(None))
    )
)

print(f"✓ Filas en order_summary: {df_order_summary.count()}")

In [0]:
# ============================================================
# GLD_ORDER_SUMMARY — Celda 5: Escritura
# ============================================================

(
    df_order_summary.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(GOLD_ORDER_SUMMARY_FQN)
)

print(f"✓ Tabla {GOLD_ORDER_SUMMARY_FQN} escrita: {df_order_summary.count()} filas")

In [0]:
TABLA_DESTINO = "gld_dataco.supply_chain.desc_ft_order_summary"   # cambia por notebook (ver tabla)
filas_procesadas = spark.table(TABLA_DESTINO).count()

spark.sql(f"""
UPDATE metadata_dataco.supply_chain.hd_etl_log
SET estado = 'EXITOSO',
    fecha_fin = current_timestamp(),
    filas_procesadas = {filas_procesadas}
WHERE log_id = '{LOG_ID}'
""")

print(f"✓ etl_log actualizado: EXITOSO ({filas_procesadas} filas en {TABLA_DESTINO})")